In [1]:
from pathlib import Path
import pandas as pd

# Path to the dataset
DATA_PATH = Path("../data/IMDB Dataset.csv")

# Check that the dataset exists
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found!\n"
        f"Download the IMDB 50K Reviews dataset from Kaggle\n"
        f"and place it here:\n{DATA_PATH.resolve()}"
    )

# Load the dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (50000, 2)


## Dataset Observations

- The dataset contains 50,000 movie reviews.
- There are two columns:
  - review: the movie review text.
  - sentiment: the sentiment label (positive or negative).
- There are no missing values.
- The dataset is balanced with an equal number of positive and negative reviews.
- Reviews contain HTML tags, punctuation, numbers, mixed casing, and inconsistent whitespace, making preprocessing necessary before tokenization and model training.

In [2]:
print("=" * 50)
print("Dataset Shape")
print("=" * 50)
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Shape
(50000, 2)

Columns:
['review', 'sentiment']

Data Types:
review       str
sentiment    str
dtype: object


In [3]:
print("Missing Values")
print(df.isnull().sum())

Missing Values
review       0
sentiment    0
dtype: int64


In [4]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 418


In [5]:
print(df["sentiment"].value_counts())

print("\nPercentage:")

print(df["sentiment"].value_counts(normalize=True) * 100)

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Percentage:
sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64


In [6]:
for i in range(3):
    print("=" * 80)
    print("Sentiment:", df.iloc[i]["sentiment"])
    print()
    print(df.iloc[i]["review"])
    print()

Sentiment: positive

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the s

In [7]:
df.describe(include="all")

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


## Basic Preprocessing

The first preprocessing step is lowercase conversion. Lowercasing reduces
vocabulary duplication by treating forms such as "Movie", "movie", and
"MOVIE" as the same token.

A limitation is that capitalization information is lost. However, for this
sentiment-analysis dataset, the reduced vocabulary is generally more useful
than preserving capitalization.

In [9]:
from preprocessor import lowercase_text

In [10]:
sample_text = "This Movie Was AMAZING, But The Ending Was Weak!"

cleaned_text = lowercase_text(sample_text)

print("Original:")
print(sample_text)

print("\nLowercase:")
print(cleaned_text)

Original:
This Movie Was AMAZING, But The Ending Was Weak!

Lowercase:
this movie was amazing, but the ending was weak!


In [11]:
sample_review = df.loc[0, "review"]

lowercase_review = lowercase_text(sample_review)

print("Original review:")
print(sample_review[:500])

print("\nLowercase review:")
print(lowercase_review[:500])

Original review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Lowercase review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of

## Regex-Based Cleaning

The preprocessing pipeline uses regular expressions to remove common forms
of noise from movie reviews:

1. HTML tags are removed because the dataset frequently contains tags such
   as `<br />`.
2. URLs are removed because their exact content usually does not help with
   sentiment classification.
3. Numbers are removed according to the assignment requirements.
4. Punctuation and special characters are removed so that the remaining
   text contains letters and spaces.
5. New lines, tabs, carriage returns, and repeated spaces are converted into
   a single space.

The order matters. URLs and HTML tags are removed before punctuation so
that parts of those structures do not remain as meaningless tokens.
Whitespace normalization is performed last.

In [13]:
import importlib
import preprocessor

importlib.reload(preprocessor)

from preprocessor import preprocess_text

In [14]:
sample_text = """
<br /><br />
This Movie was AMAZING!!! I gave it 9/10.
Visit https://example.com for more reviews.
\tIt had great acting & music!!! $$$
"""

cleaned_text = preprocess_text(sample_text)

print("Original:")
print(sample_text)

print("\nCleaned:")
print(cleaned_text)

Original:

<br /><br />
This Movie was AMAZING!!! I gave it 9/10.
Visit https://example.com for more reviews.
	It had great acting & music!!! $$$


Cleaned:
this movie was amazing i gave it visit for more reviews it had great acting music


## Stop-Word Removal

Stop words are frequent words such as "the", "is", and "and". Removing them
can reduce vocabulary size and eliminate words that may add little useful
information.

Stop-word removal is optional because common words can sometimes carry
important meaning. This is especially true in sentiment analysis.

Negation words such as "not", "no", "nor", and "never" are preserved because
removing them may reverse the meaning of a review. For example, "not good"
should not become only "good".

Stop words are removed after lowercase conversion and regex cleaning so that
the tokens consistently match the lowercase stop-word vocabulary.

In [17]:
import importlib
import preprocessor

importlib.reload(preprocessor)

from preprocessor import preprocess_text, remove_stop_words

In [18]:
sample_text = (
    "This movie is not good and the acting is very weak, "
    "but I never felt completely bored."
)

cleaned_without_stopwords = preprocess_text(
    sample_text,
    remove_stopwords=False,
)

cleaned_with_stopwords = preprocess_text(
    sample_text,
    remove_stopwords=True,
)

print("Without stop-word removal:")
print(cleaned_without_stopwords)

print("\nWith stop-word removal:")
print(cleaned_with_stopwords)

Without stop-word removal:
this movie is not good and the acting is very weak but i never felt completely bored

With stop-word removal:
movie not good acting weak never felt completely bored


In [19]:
custom_stop_words = {"this", "movie", "is"}

result = remove_stop_words(
    "this movie is genuinely excellent",
    stop_words=custom_stop_words,
)

print(result)

genuinely excellent


## Stemming

Stemming reduces related word forms to a shared stem. For example,
"connected", "connecting", and "connection" may all be reduced to
"connect".

This can reduce vocabulary size and help a model treat related word forms
as similar. However, stemming may produce forms that are not valid English
words, such as reducing "studies" to "studi".

Stemming is optional in this pipeline because it can remove useful linguistic
information. It is applied after text cleaning and stop-word removal. Stop
words are removed first because the stop-word list contains normal word forms,
not stemmed forms.

In [20]:
import importlib
import preprocessor

importlib.reload(preprocessor)

from preprocessor import preprocess_text, stem_text

In [21]:
sample_text = "connected connecting connection studies studied studying"

stemmed_text = stem_text(sample_text)

print("Original:")
print(sample_text)

print("\nStemmed:")
print(stemmed_text)

Original:
connected connecting connection studies studied studying

Stemmed:
connect connect connect studi studi studi


In [22]:
sample_review = (
    "The actors were acting brilliantly, and I really enjoyed "
    "the connected storylines. I was not disappointed!"
)

basic_result = preprocess_text(
    sample_review,
    remove_stopwords=False,
    apply_stemming=False,
)

stopword_result = preprocess_text(
    sample_review,
    remove_stopwords=True,
    apply_stemming=False,
)

stemmed_result = preprocess_text(
    sample_review,
    remove_stopwords=True,
    apply_stemming=True,
)

print("Basic preprocessing:")
print(basic_result)

print("\nWith stop-word removal:")
print(stopword_result)

print("\nWith stop-word removal and stemming:")
print(stemmed_result)

Basic preprocessing:
the actors were acting brilliantly and i really enjoyed the connected storylines i was not disappointed

With stop-word removal:
actors acting brilliantly really enjoyed connected storylines not disappointed

With stop-word removal and stemming:
actor act brilliantli realli enjoy connect storylin not disappoint


In [ ]:
original_review = df.loc[0, "review"]

processed_review = preprocess_text(
    original_review,
    remove_stopwords=True,
    apply_stemming=True,
)

print("Original review:")
print(original_review[:500])

print("\nProcessed review:")
print(processed_review[:500])

Original review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Processed review:
one review mention watch oz episod hook right exactli happen first thing struck oz brutal unflinch scene violenc set right word go trust not show faint heart timid show pull no punch regard drug sex violenc hardcor classic use word call oz nicknam given oswald maximum secur state penitentari focus mainli emerald citi experiment section prison cell glass front face inward privaci not high agenda em citi home mani aryan muslim gangsta latino christian italian i

## Tokenization

Tokenization is the process of splitting text into smaller units called
tokens. A token may be a word, character, or part of a word depending on the
tokenization method.

For example, a word-level tokenizer may convert:

"I really enjoyed this movie"

into:

["I", "really", "enjoyed", "this", "movie"]

Tokenization is important because machine-learning models cannot directly
process raw text. The text must first be divided into consistent units that
can later be counted, mapped to IDs, or converted into numerical vectors.

## Types of Tokenization

### Word-Level Tokenization

Word-level tokenization splits text into complete words.

Example:

"this movie was excellent"

becomes:

["this", "movie", "was", "excellent"]

Advantages:
- Easy to understand and implement.
- Tokens usually preserve clear meaning.
- Useful for classical NLP methods.

Disadvantages:
- Produces a large vocabulary.
- Cannot naturally handle unseen words.
- Different word forms are treated as separate tokens.


### Character-Level Tokenization

Character-level tokenization splits text into individual characters.

Example:

"movie"

becomes:

["m", "o", "v", "i", "e"]

Advantages:
- Requires a small vocabulary.
- Can represent any word.
- Handles spelling mistakes and unseen words.

Disadvantages:
- Produces much longer sequences.
- Individual characters carry less meaning than words.
- The model must learn how characters combine into meaningful units.


### Subword-Level Tokenization

Subword tokenization splits words into smaller meaningful or frequently
occurring pieces.

Example:

"unhappiness"

may become:

["un", "happi", "ness"]

Advantages:
- Handles unseen and rare words better than word-level tokenization.
- Uses a smaller vocabulary than word-level tokenization.
- Preserves more meaning than character-level tokenization.

Disadvantages:
- More complex to build and understand.
- A single word may become several tokens.
- Tokenization depends on the learned subword vocabulary.

In [24]:
sample_text = "unbelievable movies"

word_tokens = sample_text.split()
character_tokens = list(sample_text)

print("Original text:")
print(sample_text)

print("\nWord-level tokens:")
print(word_tokens)

print("\nCharacter-level tokens:")
print(character_tokens)

Original text:
unbelievable movies

Word-level tokens:
['unbelievable', 'movies']

Character-level tokens:
['u', 'n', 'b', 'e', 'l', 'i', 'e', 'v', 'a', 'b', 'l', 'e', ' ', 'm', 'o', 'v', 'i', 'e', 's']


## Preprocessing vs Tokenization

Preprocessing modifies and cleans text, while tokenization divides text into
units.

For example:

Raw text:
"This MOVIE!!!"

After preprocessing:
"this movie"

After tokenization:
["this", "movie"]

The two stages should remain separate so that the same tokenizer can be used
with different preprocessing settings.

In [25]:
import importlib
import tokenizer

importlib.reload(tokenizer)

from tokenizer import (
    whitespace_tokenize,
    regex_tokenize,
    character_tokenize,
)

In [26]:
sample_text = "I don't think this movie was great!"

whitespace_tokens = whitespace_tokenize(sample_text)
regex_tokens = regex_tokenize(sample_text)

print("Original:")
print(sample_text)

print("\nWhitespace tokens:")
print(whitespace_tokens)

print("\nRegex tokens:")
print(regex_tokens)

Original:
I don't think this movie was great!

Whitespace tokens:
['I', "don't", 'think', 'this', 'movie', 'was', 'great!']

Regex tokens:
['I', "don't", 'think', 'this', 'movie', 'was', 'great']


In [27]:
sample_word = "movie"

characters = character_tokenize(sample_word)

print(characters)

['m', 'o', 'v', 'i', 'e']


In [28]:
sample_text = "good movie"

without_spaces = character_tokenize(
    sample_text,
    include_spaces=False,
)

with_spaces = character_tokenize(
    sample_text,
    include_spaces=True,
)

print("Without spaces:")
print(without_spaces)

print("\nWith spaces:")
print(with_spaces)

Without spaces:
['g', 'o', 'o', 'd', 'm', 'o', 'v', 'i', 'e']

With spaces:
['g', 'o', 'o', 'd', ' ', 'm', 'o', 'v', 'i', 'e']


In [29]:
from preprocessor import preprocess_text
from tokenizer import regex_tokenize

sample_review = """
<br /> This MOVIE was incredible!!!
I gave it 10/10, but the ending wasn't perfect.
"""

cleaned_review = preprocess_text(
    sample_review,
    remove_stopwords=False,
    apply_stemming=False,
)

tokens = regex_tokenize(cleaned_review)

print("Original:")
print(sample_review)

print("\nCleaned:")
print(cleaned_review)

print("\nTokens:")
print(tokens)

Original:

<br /> This MOVIE was incredible!!!
I gave it 10/10, but the ending wasn't perfect.


Cleaned:
this movie was incredible i gave it but the ending wasn t perfect

Tokens:
['this', 'movie', 'was', 'incredible', 'i', 'gave', 'it', 'but', 'the', 'ending', 'wasn', 't', 'perfect']


In [30]:
original_review = df.loc[0, "review"]

cleaned_review = preprocess_text(
    original_review,
    remove_stopwords=False,
    apply_stemming=False,
)

review_tokens = regex_tokenize(cleaned_review)

print("Original review length:")
print(len(original_review))

print("\nNumber of tokens:")
print(len(review_tokens))

print("\nFirst 30 tokens:")
print(review_tokens[:30])

Original review length:
1761

Number of tokens:
313

First 30 tokens:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first']


## Custom Tokenizer

A tokenizer was implemented from scratch using two approaches.

The whitespace tokenizer splits text whenever whitespace appears. It is simple
and fast, but punctuation may remain attached to words.

The regex tokenizer extracts word-like patterns and separates punctuation from
the tokens. It also supports basic contractions such as "don't" when the input
text still contains apostrophes.

A character tokenizer was also implemented to divide text into individual
characters. Character tokenization uses a much smaller vocabulary but creates
longer token sequences.

The preprocessing and tokenization stages remain separate so that different
cleaning settings can be combined with different tokenizer implementations.